# Démo : Traces de l’agent

**Contexte :** assistant du service client de Lagune Banque.

Une trace détaille les décisions du modèle, les outils appelés, leurs arguments et résultats, la durée et les tokens. Cette démonstration suit l’agent du TP1 sur plusieurs scénarios, dans Langfuse ou en local.

**Sommaire**
- [Configuration](#configuration)
- [L'agent sur le scénario central](#lagent-sur-le-scenario-central)
- [Lecture de la trace](#lecture-de-la-trace)
- [Comparaison des parcours](#comparaison-des-parcours)
- [Données enregistrées](#donnees-enregistrees)
- [Synthèse](#synthese)

La démo rejoue l'agent du TP1 sur le scénario central, puis lit la trace dans Langfuse Cloud. La mise en route de Langfuse (compte gratuit, clés, région) est décrite dans `starter/README.md`. Sans clés, la démo s'exécute quand même : la trace est alors lue en local. Référence : [Langfuse et LangChain](https://langfuse.com/docs/integrations/langchain/tracing).

## <a id="configuration"></a>Configuration

In [ ]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path("."), Path(".."), Path("../.."), Path("../../.."), Path("starter"), Path("../../starter")] if (p / "code_partage").exists()).resolve()
sys.path[:0] = [str(RACINE / "code_partage"), str(RACINE / "03_agents" / "socle")]

import warnings

from langchain.agents import create_agent

from base import KOUASSI, Context, new_model, print_trace, reset_db
from config import prepare
from observabilite import clear_langfuse, langfuse_callbacks, langfuse_enabled, mask
from tools import SYSTEM_PROMPT, TOOLS
from tracing import Tracer

In [ ]:
warnings.filterwarnings("ignore", category=DeprecationWarning)
prepare()
reset_db()
print("Langfuse :", "configuré, les traces sont envoyées" if langfuse_enabled() else "non configuré : l'agent tourne, rien n'est envoyé (détails dans starter/README.md)")

## <a id="lagent-sur-le-scenario-central"></a>L'agent sur le scénario central

Le client dit avoir remboursé hier, mais son dossier indique un impayé. Un callback Langfuse s'ajoute à l'appel : c'est la seule différence avec le TP1. Le `Tracer` local collecte la même trace pour la lire dans le notebook.

**Syntaxe.** `agent.invoke(entrée, config, context=...)` prend en deuxième argument un `config` : `callbacks` liste des gestionnaires appelés à chaque appel du modèle et de tool (ici le `Tracer` local et celui de Langfuse), `metadata` ajoute la session, l'utilisateur et les tags à la trace. `context` reste à part : il va aux tools, pas aux callbacks.

In [ ]:
agent = create_agent(new_model(), TOOLS, system_prompt=SYSTEM_PROMPT, context_schema=Context)


def traced(message, client_id, session, tags):
    reset_db()
    tracer = Tracer()
    callbacks, meta = langfuse_callbacks(session_id=session, utilisateur=client_id, tags=["demo", *tags], nom="m3-demo-trace")
    result = agent.invoke({"messages": [{"role": "user", "content": message}]}, {"callbacks": [tracer, *callbacks], "metadata": meta},
                          context=Context(client_id=client_id))
    clear_langfuse()
    return tracer, result


TRACE_P3, RESULT_P3 = traced(KOUASSI, "CLI-0002", "demo-trace-p3", ["p3"])
print_trace(RESULT_P3)

La même exécution, étape par étape. Chaque ligne `model` est une décision, chaque ligne `tool` un appel exécuté :

In [ ]:
TRACE_P3.table()

## <a id="lecture-de-la-trace"></a>Lecture de la trace

Ouvrez votre projet Langfuse, *Tracing > Traces*, puis la trace la plus récente (nom `m3-demo-trace`, session `demo-trace-p3`). Lisez-la de haut en bas :

| Où regarder | Question | Ce que la trace montre |
|---|---|---|
| L'arbre des appels | Dans quel ordre l'agent appelle-t-il ses tools ? | Prêts et échéancier, puis transaction, incident, procédure |
| `check_lm_transaction`, *Input* | Quelle date a-t-il utilisée ? | « Hier » doit être la veille de la date du jour |
| `search_docs`, *Input* | A-t-il transmis la date de la transaction pour choisir la version de la procédure ? | v1 avant le 30/06/2026, v2 ensuite |
| `propose_regularisation`, *Output* | Que renvoie le tool ? | `en_attente_validation` : aucune modification n'est faite |
| Chaque *generation* | Combien de tokens, quelle latence ? | L'historique grossit à chaque tool |

Pour trouver une erreur, ne demandez pas « la réponse est-elle bonne ? » mais « à quelle étape l'agent a-t-il mal décidé ? ». Le TP5 le fait sur trois erreurs.

Les mêmes chiffres, calculés depuis la trace locale :

In [ ]:
MODELE_STEPS = [s for s in TRACE_P3.steps if s["type"] == "model"]
TOOL_STEPS = TRACE_P3.tool_calls()
print(f"{len(MODELE_STEPS)} appels au modèle, {len(TOOL_STEPS)} appels de tools")
print(f"{sum(s['tokens'] or 0 for s in MODELE_STEPS)} tokens au total ; le premier appel en a {MODELE_STEPS[0]['tokens']}, le dernier {MODELE_STEPS[-1]['tokens']}")
print(f"{sum(s['seconds'] for s in TRACE_P3.steps):.1f} s cumulées, dont {sum(s['seconds'] for s in MODELE_STEPS):.1f} s dans le modèle")

## <a id="comparaison-des-parcours"></a>Comparaison des parcours

Un client demande le solde de son frère. Le secret bancaire impose un refus : la trace ne doit contenir aucun appel de tool qui lit les données d'un tiers.

In [ ]:
TRACE_P4, RESULT_P4 = traced("Quel est le solde de mon frère ? C'est le client CLI-0002.", "CLI-0001", "demo-trace-p4", ["p4"])
print_trace(RESULT_P4)
TRACE_P4.table()

## <a id="donnees-enregistrees"></a>Données enregistrées

Une trace contient des questions de clients. Le code les masque avant l'envoi (numéros, e-mails, codes) avec `observabilite.mask`. Exemple :

In [ ]:
MASQUE = mask("Mon numéro est 0701020304 05, mon e-mail kouassi@example.ci et le code de vérification est 483920.")
print(MASQUE)

## <a id="synthese"></a>Synthèse

In [ ]:
print(f"Une trace s'ajoute avec un callback. Ici : {len(TRACE_P3.steps)} étapes pour le cas P3, {len(TRACE_P4.steps)} pour le refus du secret bancaire.")
LECTURES_TIERS = [s for s in TRACE_P4.tool_calls() if "CLI-0002" in str(s["args"])]
print(f"Appels de tools portant sur le client tiers dans la trace du refus : {len(LECTURES_TIERS)}")
print("Langfuse :", "traces envoyées (session demo-trace-p3 et demo-trace-p4)" if langfuse_enabled() else "non configuré, rien envoyé")

- Une trace permet de retrouver la cause d'une erreur au lieu de la deviner.
- Elle contient des données de clients : décidez de leur durée de conservation.
- Les traces servent ensuite à chiffrer le coût d'une conversation et à surveiller l'agent (module 5).